# Fleet Performance Analytics — Data Cleaning & Preparation

### Purpose
Convert the raw CSV files into clean, analysis-ready datasets.

### Workflow
**Load → Validate → Clean → Create useful metrics → Save → Verify**

The raw data is never overwritten; each section performs one defined data-preparation task.

In [1]:
# 1. Imports and project paths
from pathlib import Path
import pandas as pd

# The notebook lives inside the project's notebooks folder.
BASE_PATH = Path.cwd().parent
RAW_PATH = BASE_PATH / "data" / "raw"
PROCESSED_PATH = BASE_PATH / "data" / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Project:", BASE_PATH)
print("Raw data:", RAW_PATH)
print("Processed data:", PROCESSED_PATH)

Project: /mnt/data/fpa_work/fleet_performance_analysis
Raw data: /mnt/data/fpa_work/fleet_performance_analysis/data/raw
Processed data: /mnt/data/fpa_work/fleet_performance_analysis/data/processed


## 2. Load the raw data

We work with five core tables:
- trucks
- drivers
- trips
- maintenance
- fuel purchases

In [2]:
trucks = pd.read_csv(RAW_PATH / "trucks.csv")
drivers = pd.read_csv(RAW_PATH / "drivers.csv")
trips = pd.read_csv(RAW_PATH / "trips.csv")
maintenance = pd.read_csv(RAW_PATH / "maintenance.csv")
fuel = pd.read_csv(RAW_PATH / "fuel_purchases.csv")

for name, df in {
    "Trucks": trucks, "Drivers": drivers, "Trips": trips,
    "Maintenance": maintenance, "Fuel": fuel
}.items():
    print(f"{name:15} {len(df):,} rows × {len(df.columns)} columns")

Trucks          35 rows × 9 columns
Drivers         45 rows × 11 columns
Trips           1,500 rows × 32 columns
Maintenance     350 rows × 10 columns
Fuel            500 rows × 7 columns


## 3. Validate the raw data

Before changing anything, check missing values, duplicate IDs and basic table sizes.

This is important because cleaning should be based on evidence rather than assumptions.

In [3]:
datasets = {
    "trucks": (trucks, "truck_id"),
    "drivers": (drivers, "driver_id"),
    "trips": (trips, "trip_id"),
    "maintenance": (maintenance, "maintenance_id"),
    "fuel": (fuel, "fuel_id")
}

for name, (df, id_col) in datasets.items():
    missing = int(df.isna().sum().sum())
    duplicates = int(df[id_col].duplicated().sum())
    print(f"{name:15} missing={missing:3} | duplicate IDs={duplicates:3}")

trucks          missing=  0 | duplicate IDs=  0
drivers         missing=  0 | duplicate IDs=  0
trips           missing=714 | duplicate IDs=  0
maintenance     missing=  0 | duplicate IDs=  0
fuel            missing=  0 | duplicate IDs=  0


## 4. Fix data types and remove duplicate IDs

Dates are converted to `datetime` so we can create monthly, yearly and quarterly analysis later.

In [4]:
# Remove duplicate records using the primary key of each table.
for df, id_col in [(trucks,"truck_id"),(drivers,"driver_id"),(trips,"trip_id"),
                   (maintenance,"maintenance_id"),(fuel,"fuel_id")]:
    df.drop_duplicates(subset=id_col, inplace=True)

# Convert date columns.
trucks["purchase_date"] = pd.to_datetime(trucks["purchase_date"])
drivers["hire_date"] = pd.to_datetime(drivers["hire_date"])
trips["trip_date"] = pd.to_datetime(trips["trip_date"])
maintenance["maintenance_date"] = pd.to_datetime(maintenance["maintenance_date"])
fuel["purchase_date"] = pd.to_datetime(fuel["purchase_date"])

# Ensure the delay flag is numeric 0/1 for analysis and modelling.
trips["has_delay"] = trips["has_delay"].astype(int)

print("Dates and identifiers cleaned successfully.")

Dates and identifiers cleaned successfully.


## 5. Create trip-level analytical features

These are derived from existing fields.

**Important:** already contains `operating_cost`, so we do **not** replace it with fuel cost. That was a problem in the old V1 cleaning workflow.

In [5]:
# Fuel efficiency: distance travelled per litre consumed.
trips["fuel_efficiency_km_per_l"] = (
    trips["distance_km"] / trips["fuel_consumed_litres"]
).round(2)

# Profit margin: profit as a percentage of revenue.
trips["profit_margin_pct"] = (
    trips["profit"] / trips["revenue"] * 100
).round(2)

# Cost per kilometre uses the total operating cost.
trips["cost_per_km"] = (
    trips["operating_cost"] / trips["distance_km"]
).round(2)

# Calendar features for trend analysis.
trips["trip_year"] = trips["trip_date"].dt.year
trips["trip_month"] = trips["trip_date"].dt.month
trips["trip_month_name"] = trips["trip_date"].dt.strftime("%B")
trips["trip_quarter"] = trips["trip_date"].dt.quarter
trips["trip_day_of_week"] = trips["trip_date"].dt.day_name()

print(trips[["trip_id","fuel_efficiency_km_per_l","profit_margin_pct","cost_per_km"]].head())

    trip_id  fuel_efficiency_km_per_l  profit_margin_pct  cost_per_km
0  TRP00001                      2.81              19.50         1.16
1  TRP00002                      2.49              -9.16         1.35
2  TRP00003                      2.75              -5.23         1.40
3  TRP00004                      3.05              12.73         1.11
4  TRP00005                      2.74              11.83         1.21


## 6. Build truck and driver performance tables

We aggregate trip and maintenance activity to create management-level KPIs for each truck and driver.

In [6]:
# ---- Truck KPIs ----
truck_trip = trips.groupby("truck_id").agg(
    total_trips=("trip_id", "count"),
    total_distance_km=("distance_km", "sum"),
    total_fuel_litres=("fuel_consumed_litres", "sum"),
    total_revenue=("revenue", "sum"),
    total_operating_cost=("operating_cost", "sum"),
    total_profit=("profit", "sum"),
    avg_fuel_efficiency_km_per_l=("fuel_efficiency_km_per_l", "mean")
).reset_index()

truck_maint = maintenance.groupby("truck_id").agg(
    total_maintenance_cost=("cost", "sum"),
    maintenance_count=("maintenance_id", "count"),
    total_downtime_days=("downtime_days", "sum")
).reset_index()

trucks_clean = trucks.merge(truck_trip, on="truck_id", how="left")
trucks_clean = trucks_clean.merge(truck_maint, on="truck_id", how="left")

# Trucks without a record in an aggregate table receive zero activity.
metric_cols = [c for c in trucks_clean.columns if c.startswith(("total_", "maintenance_", "avg_"))]
trucks_clean[metric_cols] = trucks_clean[metric_cols].fillna(0)
trucks_clean["operating_cost_per_km"] = (
    trucks_clean["total_operating_cost"] / trucks_clean["total_distance_km"].replace(0, pd.NA)
).fillna(0).round(2)
trucks_clean["profit_margin_pct"] = (
    trucks_clean["total_profit"] / trucks_clean["total_revenue"].replace(0, pd.NA) * 100
).fillna(0).round(2)

# ---- Driver KPIs ----
driver_trip = trips.groupby("driver_id").agg(
    total_trips=("trip_id", "count"),
    total_distance_km=("distance_km", "sum"),
    total_revenue=("revenue", "sum"),
    total_profit=("profit", "sum"),
    total_fuel_litres=("fuel_consumed_litres", "sum"),
    total_delays=("has_delay", "sum"),
    total_delay_minutes=("delay_minutes", "sum"),
    total_duration_hours=("duration_hours", "sum"),
    avg_fuel_efficiency_km_per_l=("fuel_efficiency_km_per_l", "mean")
).reset_index()

drivers_clean = drivers.merge(driver_trip, on="driver_id", how="left")
driver_cols = [c for c in driver_trip.columns if c != "driver_id"]
drivers_clean[driver_cols] = drivers_clean[driver_cols].fillna(0)
drivers_clean["delay_rate_pct"] = (
    drivers_clean["total_delays"] / drivers_clean["total_trips"].replace(0, pd.NA) * 100
).fillna(0).round(2)
drivers_clean["avg_speed_kmh"] = (
    drivers_clean["total_distance_km"] / drivers_clean["total_duration_hours"].replace(0, pd.NA)
).fillna(0).round(1)

print("Truck KPI columns:", len(trucks_clean.columns))
print("Driver KPI columns:", len(drivers_clean.columns))

Truck KPI columns: 21
Driver KPI columns: 22


## 7. Prepare maintenance and fuel tables

Only useful date features are added. The original transaction values are preserved.

In [7]:
maintenance_clean = maintenance.copy()
maintenance_clean["maintenance_year"] = maintenance_clean["maintenance_date"].dt.year
maintenance_clean["maintenance_month"] = maintenance_clean["maintenance_date"].dt.month
maintenance_clean["maintenance_month_name"] = maintenance_clean["maintenance_date"].dt.strftime("%B")

fuel_clean = fuel.copy()
fuel_clean["purchase_year"] = fuel_clean["purchase_date"].dt.year
fuel_clean["purchase_month"] = fuel_clean["purchase_date"].dt.month
fuel_clean["purchase_month_name"] = fuel_clean["purchase_date"].dt.strftime("%B")

print("Maintenance rows:", len(maintenance_clean))
print("Fuel rows:", len(fuel_clean))

Maintenance rows: 350
Fuel rows: 500


## 8. Save the processed datasets

The raw CSVs remain untouched. These five files become the official inputs for EDA, SQL, statistics and Power BI.

In [8]:
processed = {
    "trucks_clean.csv": trucks_clean,
    "drivers_clean.csv": drivers_clean,
    "trips_clean.csv": trips,
    "maintenance_clean.csv": maintenance_clean,
    "fuel_purchases_clean.csv": fuel_clean
}

for filename, df in processed.items():
    df.to_csv(PROCESSED_PATH / filename, index=False)
    print(f"✓ {filename:28} {len(df):,} rows")

✓ trucks_clean.csv             35 rows
✓ drivers_clean.csv            45 rows
✓ trips_clean.csv              1,500 rows
✓ maintenance_clean.csv        350 rows
✓ fuel_purchases_clean.csv     500 rows


## 9. Final quality check

A simple final check confirms that the processed files exist and the main business metrics are sensible.

In [9]:
print("\nFINAL DATA CHECK")
print("-" * 40)
print(f"Trips:             {len(trips):,}")
print(f"Delay rate:        {trips['has_delay'].mean()*100:.2f}%")
print(f"Loss-making trips: {(trips['profit'] <= 0).mean()*100:.2f}%")
print(f"Revenue:           ${trips['revenue'].sum():,.2f}")
print(f"Profit:            ${trips['profit'].sum():,.2f}")
print(f"Fuel efficiency:   {trips['fuel_efficiency_km_per_l'].mean():.2f} km/L")
print("\nProcessed files:")
for path in sorted(PROCESSED_PATH.glob("*.csv")):
    print(f"✓ {path.name}")


FINAL DATA CHECK
----------------------------------------
Trips:             1,500
Delay rate:        25.87%
Loss-making trips: 23.47%
Revenue:           $877,946.32
Profit:            $94,863.98
Fuel efficiency:   2.69 km/L

Processed files:
✓ drivers_clean.csv
✓ fuel_purchases_clean.csv
✓ maintenance_clean.csv
✓ trips_clean.csv
✓ trucks_clean.csv


## Phase 2 complete

The raw data has now been transformed into analysis-ready tables.

### Next notebooks
1. **SQL Analysis** — answer business questions with SQL.
2. **EDA** — understand distributions and patterns.
3. **Statistical Analysis** — test relationships.
4. **Machine Learning ** — build leakage-safe predictive models.
5. **Power BI** — communicate the results to decision-makers.